# release16_remaining_failedfold_diagnostic_v1

Notebook-first diagnostic for the remaining release16 failed folds after the failed-fold participation target overlay.

- **Canonical baseline**: retained SOXL harness model frontier; full retention still requires fixed-split, `wf_2020`, `wf_2022`, `wf_2023_2024`, YTD alignment, and dashboard replay evidence.
- **Direct comparison baseline**: `release16_failedfold_overlay_checkpoint_screen_v1`, which partially improved `wf_2020` but still failed `wf_2020` and `wf_2023_2024` relative-return gates.
- **Reference baseline**: `soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02` as the SOXLBacktest buy/sell drawdown-pass reference.
- **Role**: `diagnose`. This card does not train, promote, or retain an actor.
- **Target windows**: remaining failed folds `wf_2020` and `wf_2023_2024`, with `wf_2022` retained as the guard fold.
- **Changed layer**: `model_selection` / failure attribution / target-to-replay diagnostics.
- **Hypothesis**: the previous overlay improved target coverage, but the selected BC/checkpoint path still does not express enough participation in the failed test states; the next valid loop needs feature/state or representation evidence before another target-floor tweak.
- **Acceptance criteria**: quantify each failed fold's remaining return gap, identify whether the gap is target coverage, train/test state transfer, BC representation, or PPO drift, and write a concrete next mechanism without promoting any actor.
- **Closest historical success**: `release16_failedfold_participation_target_overlay_v1` made a non-leaking target contract that improved `wf_2020` replay return.
- **Closest historical failure**: `release16_failedfold_overlay_checkpoint_screen_v1` still failed `wf_2020` and `wf_2023_2024`; repeating another broad target floor would be duplicate work.
- **Why this is not a duplicate**: this notebook does not change targets or rerun checkpoint training; it diagnoses why the overlay failed to clear the remaining gates.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import math
import os
import sys
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'AGENTS.md').exists():
    raise RuntimeError(f'Could not locate repo root from {Path.cwd()}')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, init_qlib, segment_frame
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'release16_remaining_failedfold_diagnostic_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

CHECKPOINT_CARD_NAME = 'release16_failedfold_overlay_checkpoint_screen_v1'
PRIOR_CHECKPOINT_CARD_NAME = 'release16_cb99_rebound_checkpoint_screen_v1'
TARGET_CARD_NAME = 'release16_failedfold_participation_target_overlay_v1'
REFERENCE_CARD_NAME = 'soxlbacktest_qlib_walkforward_baseline_v1'
REFERENCE_CANDIDATE = 'wf_pass_ytd18_15_15_02'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'

CHECKPOINT_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / CHECKPOINT_CARD_NAME
PRIOR_CHECKPOINT_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / PRIOR_CHECKPOINT_CARD_NAME
TARGET_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / TARGET_CARD_NAME
REFERENCE_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / REFERENCE_CARD_NAME

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
FAILED_FOLDS = ('wf_2020', 'wf_2023_2024')
GUARD_FOLDS = ('wf_2022',)
WALK_FORWARD_WINDOWS = {
    'wf_2020': {'target_fold': 'wf_2020_train', 'train': ('2010-03-11', '2018-12-31'), 'valid': ('2019-01-01', '2019-12-31'), 'test': ('2020-01-01', '2020-12-31')},
    'wf_2022': {'target_fold': 'wf_2022_train', 'train': ('2010-03-11', '2020-12-31'), 'valid': ('2021-01-01', '2021-12-31'), 'test': ('2022-01-01', '2022-12-31')},
    'wf_2023_2024': {'target_fold': 'wf_2023_2024_train', 'train': ('2010-03-11', '2022-12-31'), 'valid': ('2023-01-01', '2023-12-31'), 'test': ('2024-01-01', '2024-12-31')},
}

SIDE_CARS = {
    'fold_gate_gap_summary': CARD / 'fold_gate_gap_summary.csv',
    'monthly_path_attribution': CARD / 'monthly_overlay_vs_prior_vs_reference.csv',
    'missed_up_days': CARD / 'missed_up_days_after_overlay.csv',
    'missed_up_days_summary': CARD / 'missed_up_days_summary.csv',
    'target_to_selected_exposure_gap': CARD / 'target_to_selected_exposure_gap.csv',
    'test_state_trigger_overlap': CARD / 'test_state_trigger_overlap.csv',
    'bc_representation_summary': CARD / 'bc_representation_summary.csv',
    'selection': CARD / 'selection.json',
    'summary': CARD / 'summary.json',
    'conclusion': CARD / 'conclusion.md',
}


def rel(path: Path) -> str:
    return str(path.relative_to(ROOT))


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, ensure_ascii=False, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}


def return_pct(nav: pd.Series) -> float:
    nav = pd.to_numeric(nav, errors='coerce').dropna()
    if nav.empty or float(nav.iloc[0]) == 0.0:
        return float('nan')
    return float((nav.iloc[-1] / nav.iloc[0] - 1.0) * 100.0)


def max_drawdown_pct(nav: pd.Series) -> float:
    nav = pd.to_numeric(nav, errors='coerce').dropna()
    if nav.empty:
        return float('nan')
    peak = nav.cummax()
    return float(((peak - nav) / peak).max() * 100.0)


def normalize_date_column(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    if 'datetime' in out.columns:
        out['datetime'] = pd.to_datetime(out['datetime']).dt.normalize()
    elif 'date' in out.columns:
        out['datetime'] = pd.to_datetime(out['date']).dt.normalize()
    else:
        raise ValueError('daily frame needs datetime or date')
    return out

print('card', rel(CARD))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


card runtime/model_selection/card/release16_remaining_failedfold_diagnostic_v1


In [2]:
checkpoint_summary = load_json(CHECKPOINT_CARD / 'summary.json')
checkpoint_selection = pd.read_csv(CHECKPOINT_CARD / 'checkpoint_selection_by_fold.csv')
prior_selection = pd.read_csv(PRIOR_CHECKPOINT_CARD / 'checkpoint_selection_by_fold.csv')
checkpoint_delta = pd.read_csv(CHECKPOINT_CARD / 'overlay_vs_source_checkpoint_delta.csv')
bc_slice = pd.read_csv(CHECKPOINT_CARD / 'bc_slice_metrics.csv')
candidate_metrics = pd.read_csv(CHECKPOINT_CARD / 'checkpoint_candidate_metrics.csv')
metric_basis = pd.read_csv(CHECKPOINT_CARD / 'metric_basis_reconciliation.csv')
target_contract = pd.read_csv(TARGET_CARD / 'combined_target_contract.csv', parse_dates=['datetime'])
target_overlay_rows = pd.read_csv(TARGET_CARD / 'target_overlay_rows.csv', parse_dates=['datetime'])
target_distribution = pd.read_csv(TARGET_CARD / 'target_distribution_before_after.csv')
reference_metrics = pd.read_csv(REFERENCE_CARD / 'candidate_metrics.csv')
reference_daily = pd.read_csv(REFERENCE_CARD / 'candidate_daily.csv')

assert checkpoint_summary.get('decision') == 'release16_failedfold_overlay_checkpoint_metric_screen_partial_pass_no_promotion', checkpoint_summary.get('decision')
assert set(checkpoint_summary.get('failed_folds', [])) == set(FAILED_FOLDS), checkpoint_summary.get('failed_folds')
print('checkpoint decision', checkpoint_summary.get('decision'))
print('failed folds', checkpoint_summary.get('failed_folds'))
print('target rows', len(target_contract), 'overlay rows', len(target_overlay_rows))


checkpoint decision release16_failedfold_overlay_checkpoint_metric_screen_partial_pass_no_promotion
failed folds ['wf_2020', 'wf_2023_2024']
target rows 3731 overlay rows 892


In [3]:
# Gate gap summary: current selected path, prior release16 path, and SOXLBacktest reference.
rows = []
for row in checkpoint_selection.to_dict('records'):
    fold = str(row['fold'])
    buyhold = float(pd.to_numeric(candidate_metrics[candidate_metrics['fold'].eq(fold)]['test_buy_and_hold_return'], errors='coerce').dropna().iloc[0])
    required_return = buyhold + TARGET_RELATIVE_RETURN_PP
    prior = prior_selection[prior_selection['fold'].eq(fold)].iloc[0]
    delta = checkpoint_delta[checkpoint_delta['fold'].eq(fold)].iloc[0]
    rows.append({
        'fold': fold,
        'selected_candidate': row['selected_candidate'],
        'selected_return_pct': float(row['selected_return']),
        'selected_relative_return_pp': float(row['selected_relative_return']),
        'selected_max_dd_pct': float(row['selected_max_dd']),
        'selected_avg_pos': float(row['selected_avg_pos']),
        'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
        'target_max_dd_pct': TARGET_MAX_DD_PCT,
        'buyhold_return_pct': buyhold,
        'required_return_for_plus30_relative_pct': required_return,
        'return_gap_to_plus30_relative_pp': max(0.0, required_return - float(row['selected_return'])),
        'relative_gap_to_target_pp': max(0.0, TARGET_RELATIVE_RETURN_PP - float(row['selected_relative_return'])),
        'maxdd_gap_to_target_pp': max(0.0, float(row['selected_max_dd']) - TARGET_MAX_DD_PCT),
        'candidate_gate': bool(row['selected_candidate_gate']),
        'prior_return_pct': float(prior['selected_return']),
        'prior_relative_return_pp': float(prior['selected_relative_return']),
        'prior_max_dd_pct': float(prior['selected_max_dd']),
        'overlay_return_delta_pct': float(delta['return_delta_pct']),
        'overlay_relative_delta_pp': float(delta['relative_return_delta_pp']),
        'overlay_maxdd_delta_pp': float(delta['max_dd_delta_pct']),
        'overlay_avg_pos_delta': float(delta['avg_pos_delta']),
        'reference_candidate': row['reference_candidate'],
        'reference_return_pct': float(row['reference_return_pct']),
        'reference_relative_return_pp': float(row['reference_relative_return_pp']),
        'reference_max_dd_pct': float(row['reference_max_dd_pct']),
    })
fold_gate_gap = pd.DataFrame(rows)
fold_gate_gap.to_csv(SIDE_CARS['fold_gate_gap_summary'], index=False)
display(fold_gate_gap)


,fold,selected_candidate,selected_return_pct,selected_relative_return_pp,selected_max_dd_pct,selected_avg_pos,target_relative_return_pp,target_max_dd_pct,buyhold_return_pct,required_return_for_plus30_relative_pct,...,prior_relative_return_pp,prior_max_dd_pct,overlay_return_delta_pct,overlay_relative_delta_pp,overlay_maxdd_delta_pp,overlay_avg_pos_delta,reference_candidate,reference_return_pct,reference_relative_return_pp,reference_max_dd_pct
0,wf_2020,bc_initialization,69.514153,11.420582,16.652151,0.230203,30.0,20.0,58.093571,88.093571,...,-15.032278,14.961639,26.452861,26.452861,1.690512,0.005757,wf_pass_ytd18_15_15_02,29.156603,-30.618705,12.785207
1,wf_2022,bc_initialization,-14.873092,71.714980,15.617043,0.081384,30.0,20.0,-86.588072,-56.588072,...,71.714980,15.617043,0.000000,0.000000,0.000000,0.000000,wf_pass_ytd18_15_15_02,-16.044551,70.543521,18.514585
2,wf_2023_2024,bc_initialization,3.049745,3.406380,5.546722,0.083241,30.0,20.0,-0.356635,29.643365,...,4.707959,5.618566,-1.301579,-1.301579,-0.071845,-0.002320,wf_pass_ytd18_15_15_02,4.505779,7.109208,1.061804


In [4]:
def load_checkpoint_daily(card: Path, fold: str, candidate: str = 'bc_initialization') -> pd.DataFrame:
    path = card / 'official_daily_paths' / f'{fold}__{candidate}_official_daily_path.csv'
    frame = normalize_date_column(pd.read_csv(path))
    return frame


def reference_daily_for_fold(fold: str) -> pd.DataFrame:
    frame = reference_daily[
        reference_daily['candidate'].astype(str).eq(REFERENCE_CANDIDATE)
        & reference_daily['window'].astype(str).eq(fold)
    ].copy()
    frame = normalize_date_column(frame)
    if 'strategy_nav' not in frame.columns:
        raise ValueError('reference daily missing strategy_nav')
    return frame


def monthly_metrics(frame: pd.DataFrame, *, fold: str, source: str) -> pd.DataFrame:
    work = normalize_date_column(frame).sort_values('datetime').copy()
    work['month'] = work['datetime'].dt.to_period('M').astype(str)
    out_rows = []
    for month, group in work.groupby('month', sort=True):
        out_rows.append({
            'fold': fold,
            'source': source,
            'month': month,
            'rows': int(len(group)),
            'start': group['datetime'].min().date().isoformat(),
            'end': group['datetime'].max().date().isoformat(),
            'return_pct': return_pct(group['strategy_nav']),
            'buyhold_return_pct': return_pct(group['buyhold_nav']),
            'relative_return_pp': return_pct(group['strategy_nav']) - return_pct(group['buyhold_nav']),
            'max_dd_pct': max_drawdown_pct(group['strategy_nav']),
            'avg_position': float(pd.to_numeric(group.get('position', pd.Series(dtype=float)), errors='coerce').mean()),
        })
    return pd.DataFrame(out_rows)

monthly_frames = []
path_frames = []
for fold in FAILED_FOLDS:
    overlay = load_checkpoint_daily(CHECKPOINT_CARD, fold)
    prior = load_checkpoint_daily(PRIOR_CHECKPOINT_CARD, fold)
    ref = reference_daily_for_fold(fold)
    overlay['source'] = 'overlay_selected_bc'
    prior['source'] = 'prior_release16_bc'
    ref['source'] = 'soxlbacktest_wfpass_reference'
    path_frames.extend([overlay.assign(fold=fold), prior.assign(fold=fold), ref.assign(fold=fold)])
    monthly_frames.extend([
        monthly_metrics(overlay, fold=fold, source='overlay_selected_bc'),
        monthly_metrics(prior, fold=fold, source='prior_release16_bc'),
        monthly_metrics(ref, fold=fold, source='soxlbacktest_wfpass_reference'),
    ])
monthly_attr = pd.concat(monthly_frames, ignore_index=True)
monthly_attr.to_csv(SIDE_CARS['monthly_path_attribution'], index=False)
display(monthly_attr.head(20))


,fold,source,month,rows,start,end,return_pct,buyhold_return_pct,relative_return_pp,max_dd_pct,avg_position
0,wf_2020,overlay_selected_bc,2020-01,21,2020-01-02,2020-01-31,0.000000,-16.560487,16.560487,0.000000,0.212955
1,wf_2020,overlay_selected_bc,2020-02,19,2020-02-03,2020-02-28,0.000000,-18.943494,18.943494,0.000000,0.139437
2,wf_2020,overlay_selected_bc,2020-03,22,2020-03-02,2020-03-31,-0.023369,-57.949402,57.926033,1.587954,0.281928
3,wf_2020,overlay_selected_bc,2020-04,21,2020-04-01,2020-04-30,11.617096,64.979221,-53.362125,3.529739,0.382700
4,wf_2020,overlay_selected_bc,2020-05,20,2020-05-01,2020-05-29,3.375891,38.712863,-35.336972,3.062721,0.102938
5,wf_2020,overlay_selected_bc,2020-06,22,2020-06-01,2020-06-30,5.834458,23.659654,-17.825197,6.734797,0.411648
6,wf_2020,overlay_selected_bc,2020-07,22,2020-07-01,2020-07-31,9.794691,25.601641,-15.806951,2.611979,0.254163
7,wf_2020,overlay_selected_bc,2020-08,21,2020-08-03,2020-08-31,2.872603,11.296308,-8.423705,1.730849,0.352462
8,wf_2020,overlay_selected_bc,2020-09,21,2020-09-01,2020-09-30,-2.567446,-10.260865,7.693419,16.652151,0.303784
9,wf_2020,overlay_selected_bc,2020-10,22,2020-10-01,2020-10-30,3.149864,-7.422562,10.572426,6.120333,0.097822


In [5]:
# Low-position up-day audit after the overlay.
missed_rows = []
path_by_source = pd.concat(path_frames, ignore_index=True)
for fold in FAILED_FOLDS:
    selected = path_by_source[(path_by_source['fold'].eq(fold)) & (path_by_source['source'].eq('overlay_selected_bc'))].copy().sort_values('datetime')
    prior = path_by_source[(path_by_source['fold'].eq(fold)) & (path_by_source['source'].eq('prior_release16_bc'))][['datetime', 'position']].rename(columns={'position': 'prior_position'})
    ref = path_by_source[(path_by_source['fold'].eq(fold)) & (path_by_source['source'].eq('soxlbacktest_wfpass_reference'))][['datetime', 'position']].rename(columns={'position': 'reference_position'})
    selected['soxl_daily_return'] = selected['price'].astype(float).pct_change().fillna(0.0)
    selected['selected_prev_position'] = selected['position'].astype(float).shift(1).fillna(selected['position'].astype(float))
    merged = selected.merge(prior, on='datetime', how='left').merge(ref, on='datetime', how='left')
    merged['prior_prev_position'] = merged['prior_position'].astype(float).shift(1).fillna(merged['prior_position'].astype(float))
    merged['reference_prev_position'] = merged['reference_position'].astype(float).shift(1).fillna(merged['reference_position'].astype(float))
    low_threshold = 0.20 if fold == 'wf_2020' else 0.15
    up_threshold = 0.04
    screen = merged[(merged['soxl_daily_return'].ge(up_threshold)) & (merged['selected_prev_position'].lt(low_threshold))].copy()
    screen['selected_low_position_threshold'] = low_threshold
    screen['up_day_threshold'] = up_threshold
    screen['incremental_return_if_30pct'] = (0.30 - screen['selected_prev_position']).clip(lower=0.0) * screen['soxl_daily_return'] * 100.0
    screen['incremental_return_if_50pct'] = (0.50 - screen['selected_prev_position']).clip(lower=0.0) * screen['soxl_daily_return'] * 100.0
    screen['incremental_return_if_reference_pos'] = (screen['reference_prev_position'].fillna(0.0) - screen['selected_prev_position']).clip(lower=0.0) * screen['soxl_daily_return'] * 100.0
    screen['fold'] = fold
    missed_rows.append(screen)
missed_up_days = pd.concat(missed_rows, ignore_index=True) if missed_rows else pd.DataFrame()
missed_up_days.to_csv(SIDE_CARS['missed_up_days'], index=False)
missed_summary = missed_up_days.groupby('fold').agg(
    days=('datetime', 'size'),
    mean_selected_prev_position=('selected_prev_position', 'mean'),
    mean_reference_prev_position=('reference_prev_position', 'mean'),
    sum_soxl_up_day_return_pp=('soxl_daily_return', lambda s: float(s.sum() * 100.0)),
    mean_soxl_up_day_return_pp=('soxl_daily_return', lambda s: float(s.mean() * 100.0)),
    sum_incremental_if_30pct_pp=('incremental_return_if_30pct', 'sum'),
    sum_incremental_if_50pct_pp=('incremental_return_if_50pct', 'sum'),
    sum_incremental_if_reference_pos_pp=('incremental_return_if_reference_pos', 'sum'),
).reset_index()
missed_summary.to_csv(SIDE_CARS['missed_up_days_summary'], index=False)
display(missed_summary)
display(missed_up_days.sort_values(['fold', 'soxl_daily_return'], ascending=[True, False]).head(20))


,fold,days,mean_selected_prev_position,mean_reference_prev_position,sum_soxl_up_day_return_pp,mean_soxl_up_day_return_pp,sum_incremental_if_30pct_pp,sum_incremental_if_50pct_pp,sum_incremental_if_reference_pos_pp
0,wf_2020,47,0.101854,0.186874,376.802301,8.01707,74.664897,150.025357,35.112001
1,wf_2023_2024,66,0.069736,0.019820,490.291555,7.42866,113.514768,211.573079,4.097385


,datetime,strategy_nav,price,buyhold_nav,position,excess,excess_pct,action,source,fold,...,selected_prev_position,prior_position,reference_position,prior_prev_position,reference_prev_position,selected_low_position_threshold,up_day_threshold,incremental_return_if_30pct,incremental_return_if_50pct,incremental_return_if_reference_pos
10,2020-03-13,1.000000e+06,8.661333,4.449924e+05,0.098928,555007.564130,124.722921,HOLD,overlay_selected_bc,wf_2020,...,0.098775,0.240484,0.1591,0.240171,0.1591,0.2,0.04,5.497985,10.962506,1.648230
9,2020-03-10,1.000000e+06,11.773333,6.048773e+05,0.109579,395122.679241,65.322780,HOLD,overlay_selected_bc,wf_2020,...,0.107068,0.228652,0.1591,0.225000,0.1591,0.2,0.04,3.683250,7.501444,0.993333
8,2020-03-04,1.000000e+06,15.475333,7.950746e+05,0.110192,204925.367050,25.774356,HOLD,overlay_selected_bc,wf_2020,...,0.103215,0.240765,0.1591,0.227637,0.1591,0.2,0.04,2.880486,5.808038,0.818026
15,2020-05-18,1.078694e+06,9.232667,4.743458e+05,0.105794,604348.113084,127.406654,SELL,overlay_selected_bc,wf_2020,...,0.097053,0.101944,0.1336,0.093488,0.1336,0.2,0.04,2.839823,5.638410,0.511400
37,2020-11-05,1.614854e+06,22.343332,1.147931e+06,0.097822,466922.433984,40.675126,HOLD,overlay_selected_bc,wf_2020,...,0.097822,0.097279,0.2691,0.097279,0.2691,0.2,0.04,2.666344,5.303965,2.258831
16,2020-05-20,1.089536e+06,10.159333,5.219550e+05,0.102364,567580.869403,108.741334,HOLD,overlay_selected_bc,wf_2020,...,0.102687,0.275000,0.1336,0.098938,0.1336,0.2,0.04,2.243796,4.518151,0.351532
39,2020-11-11,1.630470e+06,22.804001,1.171599e+06,0.097822,458871.375979,39.166254,HOLD,overlay_selected_bc,wf_2020,...,0.097822,0.097279,0.2691,0.097279,0.2691,0.2,0.04,2.203015,4.382299,1.866316
36,2020-11-04,1.526601e+06,19.740000,1.014180e+06,0.097822,512420.613733,50.525609,SELL,overlay_selected_bc,wf_2020,...,0.097822,0.097279,0.2691,0.097279,0.2691,0.2,0.04,2.119018,4.215209,1.795156
11,2020-03-17,1.000000e+06,5.847333,3.004178e+05,0.275000,699582.164841,232.869718,HOLD,overlay_selected_bc,wf_2020,...,0.099748,0.275000,0.1591,0.242163,0.1591,0.2,0.04,1.990467,3.978431,0.589946
7,2020-03-02,1.000000e+06,15.020000,7.716810e+05,0.109670,228318.952111,29.587218,HOLD,overlay_selected_bc,wf_2020,...,0.105652,0.239791,0.1591,0.232249,0.1591,0.2,0.04,1.924524,3.905012,0.529270


In [6]:
# Target exposure versus selected test exposure. This checks whether the train target lift actually becomes test exposure.
exposure_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    target_fold = spec['target_fold']
    train_start, train_end = pd.Timestamp(spec['train'][0]), pd.Timestamp(spec['train'][1])
    train_targets = target_contract[target_contract['fold'].astype(str).eq(target_fold) & target_contract['datetime'].between(train_start, train_end)].copy()
    selected_path = load_checkpoint_daily(CHECKPOINT_CARD, fold_name)
    prior_path = load_checkpoint_daily(PRIOR_CHECKPOINT_CARD, fold_name)
    target_pos = pd.to_numeric(train_targets['risk_target_position'], errors='coerce')
    selected_pos = pd.to_numeric(selected_path['position'], errors='coerce')
    prior_pos = pd.to_numeric(prior_path['position'], errors='coerce')
    exposure_rows.append({
        'fold': fold_name,
        'target_fold': target_fold,
        'target_rows': int(len(train_targets)),
        'target_mean_pos': float(target_pos.mean()),
        'target_share_ge30': float(target_pos.ge(0.30).mean()),
        'target_share_ge50': float(target_pos.ge(0.50).mean()),
        'target_share_ge65': float(target_pos.ge(0.65).mean()),
        'test_rows': int(len(selected_path)),
        'selected_mean_pos': float(selected_pos.mean()),
        'selected_share_ge30': float(selected_pos.ge(0.30).mean()),
        'selected_share_ge50': float(selected_pos.ge(0.50).mean()),
        'selected_share_ge65': float(selected_pos.ge(0.65).mean()),
        'prior_mean_pos': float(prior_pos.mean()),
        'selected_minus_prior_mean_pos': float(selected_pos.mean() - prior_pos.mean()),
        'target_minus_selected_mean_pos': float(target_pos.mean() - selected_pos.mean()),
        'target_ge50_minus_selected_ge50': float(target_pos.ge(0.50).mean() - selected_pos.ge(0.50).mean()),
    })
exposure_gap = pd.DataFrame(exposure_rows)
exposure_gap.to_csv(SIDE_CARS['target_to_selected_exposure_gap'], index=False)
display(exposure_gap)


,fold,target_fold,target_rows,target_mean_pos,target_share_ge30,target_share_ge50,target_share_ge65,test_rows,selected_mean_pos,selected_share_ge30,selected_share_ge50,selected_share_ge65,prior_mean_pos,selected_minus_prior_mean_pos,target_minus_selected_mean_pos,target_ge50_minus_selected_ge50
0,wf_2020,wf_2020_train,2218,0.364056,0.417493,0.301623,0.296664,252,0.230203,0.253968,0.071429,0.019841,0.224446,0.005757,0.133853,0.230195
1,wf_2022,wf_2022_train,253,0.200429,0.162055,0.090909,0.047431,250,0.081384,0.000000,0.000000,0.000000,0.081384,0.000000,0.119044,0.090909
2,wf_2023_2024,wf_2023_2024_train,756,0.236502,0.322751,0.300265,0.034392,251,0.083241,0.000000,0.000000,0.000000,0.085560,-0.002320,0.153262,0.300265


In [7]:
# Test-state proxy overlap using the profile feature frame. This is diagnostic only and does not create labels.
profile = load_profile(PROFILE_NAME)
init_qlib(QLIB_DATA_PATH)
dataset = build_dataset(
    train_seg=('2010-03-11', '2022-12-31'),
    valid_seg=('2023-01-01', '2023-12-31'),
    test_seg=('2024-01-01', '2024-12-31'),
    feature_packs=tuple(profile.feature.packs),
    profile_name=profile.name,
)
feature_parts = []
for name in ('train', 'valid', 'test'):
    frame = segment_frame(dataset, name).copy()
    frame = frame.reset_index().rename(columns={'index': 'datetime'})
    if 'datetime' not in frame.columns:
        frame = frame.rename(columns={frame.columns[0]: 'datetime'})
    frame['datetime'] = pd.to_datetime(frame['datetime']).dt.normalize()
    frame['segment'] = name
    feature_parts.append(frame)
features = pd.concat(feature_parts, ignore_index=True).drop_duplicates('datetime', keep='last')

trigger_rows = []
for fold in FAILED_FOLDS:
    spec = WALK_FORWARD_WINDOWS[fold]
    selected = load_checkpoint_daily(CHECKPOINT_CARD, fold).sort_values('datetime').copy()
    selected['soxl_daily_return'] = selected['price'].astype(float).pct_change().fillna(0.0)
    selected['prev_position'] = selected['position'].astype(float).shift(1).fillna(selected['position'].astype(float))
    feat = features[features['datetime'].between(pd.Timestamp(spec['test'][0]), pd.Timestamp(spec['test'][1]))].copy()
    work = selected.merge(feat, on='datetime', how='left')
    if fold == 'wf_2020':
        # Feature columns for pre-2020 target rows are absent; use a replay-path recovery proxy for 2020.
        prior_nav = work['strategy_nav'].astype(float).shift(1)
        work['strategy_dd_prev_proxy'] = (prior_nav / prior_nav.cummax() - 1.0).fillna(0.0)
        work['mom5_prev_proxy'] = work['price'].astype(float).pct_change(5).shift(1).fillna(0.0)
        trigger = work['strategy_dd_prev_proxy'].le(-0.05) & work['mom5_prev_proxy'].ge(0.02)
        trigger_name = 'wf2020_path_recovery_proxy'
    else:
        positive_momentum = work['mom_5'].astype(float).ge(0.02) & work['mom_21'].astype(float).ge(0.00)
        feature_momentum = work['mom_21'].astype(float).ge(0.12) | work['relative_strength_21'].astype(float).ge(0.04)
        low_vol = work['realized_vol_21'].astype(float).le(1.50)
        non_stress = work['regime_risk'].astype(float).le(0.75)
        trigger = non_stress & low_vol & (positive_momentum | feature_momentum)
        trigger_name = 'wf2023_profile_momentum_lowvol_proxy'
    work['test_state_trigger'] = trigger.fillna(False).astype(bool)
    work['trigger_name'] = trigger_name
    work['low_position_up_day'] = work['soxl_daily_return'].ge(0.04) & work['prev_position'].lt(0.20)
    trigger_days = work[work['test_state_trigger']]
    low_up = work[work['low_position_up_day']]
    overlap = work[work['test_state_trigger'] & work['low_position_up_day']]
    trigger_rows.append({
        'fold': fold,
        'trigger_name': trigger_name,
        'test_rows': int(len(work)),
        'trigger_days': int(len(trigger_days)),
        'trigger_share': float(len(trigger_days) / len(work)) if len(work) else 0.0,
        'trigger_mean_position': float(trigger_days['position'].mean()) if not trigger_days.empty else np.nan,
        'trigger_mean_prev_position': float(trigger_days['prev_position'].mean()) if not trigger_days.empty else np.nan,
        'trigger_soxl_return_sum_pp': float(trigger_days['soxl_daily_return'].sum() * 100.0) if not trigger_days.empty else 0.0,
        'low_position_up_days': int(len(low_up)),
        'low_position_up_days_trigger_overlap': int(len(overlap)),
        'low_position_up_trigger_overlap_share': float(len(overlap) / len(low_up)) if len(low_up) else 0.0,
        'overlap_mean_prev_position': float(overlap['prev_position'].mean()) if not overlap.empty else np.nan,
        'overlap_soxl_return_sum_pp': float(overlap['soxl_daily_return'].sum() * 100.0) if not overlap.empty else 0.0,
    })
test_state_overlap = pd.DataFrame(trigger_rows)
test_state_overlap.to_csv(SIDE_CARS['test_state_trigger_overlap'], index=False)
display(test_state_overlap)


[92075:MainThread](2026-06-14 23:29:21,187) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[92075:MainThread](2026-06-14 23:29:21,594) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[92075:MainThread](2026-06-14 23:29:21,595) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


[92075:MainThread](2026-06-14 23:29:22,805) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[92075:MainThread](2026-06-14 23:29:22,805) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[92075:MainThread](2026-06-14 23:29:22,805) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


,fold,trigger_name,test_rows,trigger_days,trigger_share,trigger_mean_position,trigger_mean_prev_position,trigger_soxl_return_sum_pp,low_position_up_days,low_position_up_days_trigger_overlap,low_position_up_trigger_overlap_share,overlap_mean_prev_position,overlap_soxl_return_sum_pp
0,wf_2020,wf2020_path_recovery_proxy,252,13,0.051587,0.206257,0.206586,22.800234,47,5,0.106383,0.097959,41.064185
1,wf_2023_2024,wf2023_profile_momentum_lowvol_proxy,251,141,0.561753,0.078263,0.075980,165.875150,66,41,0.621212,0.074964,288.867503


In [8]:
# BC representation summary from existing slice metrics and exposure transfer.
bc_rows = []
for fold in WALK_FORWARD_WINDOWS:
    fold_slices = bc_slice[bc_slice['fold'].astype(str).eq(fold)].copy()
    all_slice = fold_slices[fold_slices['slice'].eq('all')]
    downtrend = fold_slices[fold_slices['slice'].eq('is_downtrend_state')]
    release = fold_slices[fold_slices['slice'].eq('release_gate_state')]
    row = {'fold': fold}
    if not all_slice.empty:
        r = all_slice.iloc[0]
        row.update({
            'bc_all_accuracy': float(r['accuracy']),
            'bc_all_samples': int(r['samples']),
            'label_dominant_action': int(r['label_dominant_action']),
            'label_dominant_share': float(r['label_dominant_share']),
            'predicted_dominant_action': int(r['predicted_dominant_action']),
            'predicted_dominant_share': float(r['predicted_dominant_share']),
            'predicted_cash_heavier_than_label': bool(int(r['predicted_dominant_action']) <= int(r['label_dominant_action']) and float(r['predicted_dominant_share']) > float(r['label_dominant_share'])),
        })
    if not downtrend.empty:
        row['bc_downtrend_accuracy'] = float(downtrend.iloc[0]['accuracy']) if pd.notna(downtrend.iloc[0]['accuracy']) else np.nan
    if not release.empty:
        row['bc_release_accuracy'] = float(release.iloc[0]['accuracy']) if pd.notna(release.iloc[0]['accuracy']) else np.nan
    exp = exposure_gap[exposure_gap['fold'].eq(fold)].iloc[0]
    row.update({
        'target_share_ge50': float(exp['target_share_ge50']),
        'selected_share_ge50': float(exp['selected_share_ge50']),
        'target_mean_pos': float(exp['target_mean_pos']),
        'selected_mean_pos': float(exp['selected_mean_pos']),
        'selected_minus_prior_mean_pos': float(exp['selected_minus_prior_mean_pos']),
    })
    bc_rows.append(row)
bc_representation = pd.DataFrame(bc_rows)
bc_representation.to_csv(SIDE_CARS['bc_representation_summary'], index=False)
display(bc_representation)


,fold,bc_all_accuracy,bc_all_samples,label_dominant_action,label_dominant_share,predicted_dominant_action,predicted_dominant_share,predicted_cash_heavier_than_label,bc_downtrend_accuracy,bc_release_accuracy,target_share_ge50,selected_share_ge50,target_mean_pos,selected_mean_pos,selected_minus_prior_mean_pos
0,wf_2020,0.738058,1905,4,0.352231,4,0.456693,True,0.864434,0.781250,0.301623,0.071429,0.364056,0.230203,0.005757
1,wf_2022,0.948617,253,0,0.418972,0,0.415020,False,0.983051,1.000000,0.090909,0.000000,0.200429,0.081384,0.000000
2,wf_2023_2024,0.708995,756,0,0.461640,0,0.572751,True,0.776151,0.909091,0.300265,0.000000,0.236502,0.083241,-0.002320


In [9]:
# Decision and next mechanism.
failed_gate = fold_gate_gap[fold_gate_gap['fold'].isin(FAILED_FOLDS)].copy()
wf2020 = failed_gate[failed_gate['fold'].eq('wf_2020')].iloc[0]
wf2023 = failed_gate[failed_gate['fold'].eq('wf_2023_2024')].iloc[0]
wf2020_improved_not_passed = bool(wf2020['overlay_relative_delta_pp'] > 20.0 and not bool(wf2020['candidate_gate']))
wf2023_regressed = bool(wf2023['overlay_relative_delta_pp'] < 0.0)
no_drawdown_blocker = bool(failed_gate['maxdd_gap_to_target_pp'].max() <= 0.0)
exposure_gap_failed = exposure_gap[exposure_gap['fold'].isin(FAILED_FOLDS)].copy()
selected_underexpresses_target = bool((exposure_gap_failed['target_ge50_minus_selected_ge50'] > 0.20).any())

if wf2020_improved_not_passed and wf2023_regressed and selected_underexpresses_target:
    decision = 'release16_overlay_failure_is_state_transfer_and_representation_gap_no_promotion'
    next_mechanism = 'mine_failedfold_test_state_features_then_rebuild_representation_or_reward_objective_before_training'
elif no_drawdown_blocker:
    decision = 'release16_overlay_remaining_failure_is_relative_return_gap_no_promotion'
    next_mechanism = 'diagnose_feature_state_overlap_before_another_target_revision'
else:
    decision = 'release16_overlay_remaining_failure_is_joint_return_drawdown_gap_no_promotion'
    next_mechanism = 'revisit_joint_drawdown_participation_objective_before_training'

metrics_for_frontier = []
for row in checkpoint_selection.to_dict('records'):
    metrics_for_frontier.append({
        'window': row['fold'],
        'split_type': 'walk_forward_test',
        'return_pct': float(row['selected_return']),
        'max_dd_pct': float(row['selected_max_dd']),
        'relative_return_pct': float(row['selected_relative_return']),
        'avg_pos': float(row['selected_avg_pos']),
    })

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'canonical_baseline': 'retained_soxl_harness_model_frontier',
    'direct_comparison_baseline': CHECKPOINT_CARD_NAME,
    'source_checkpoint_screen': CHECKPOINT_CARD_NAME,
    'source_target_experiment': TARGET_CARD_NAME,
    'reference_baseline': f'{REFERENCE_CARD_NAME}:{REFERENCE_CANDIDATE}',
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'failure_attribution', 'target_to_replay_diagnostic'],
    'target_window': list(FAILED_FOLDS),
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'failed_folds': list(FAILED_FOLDS),
    'guard_folds': list(GUARD_FOLDS),
    'metrics': metrics_for_frontier,
    'wf2020_improved_not_passed': wf2020_improved_not_passed,
    'wf2023_regressed_after_overlay': wf2023_regressed,
    'no_drawdown_blocker': no_drawdown_blocker,
    'selected_underexpresses_target': selected_underexpresses_target,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'primary_blocker': 'walkforward_relative_return_gap',
    'next_mechanism': next_mechanism,
    'sidecars': {key: rel(value) for key, value in SIDE_CARS.items() if key not in {'selection', 'summary', 'conclusion'}},
    'summary_points': [
        'wf_2020 improved materially after the overlay but still misses the +30pp relative-return gate.',
        'wf_2023_2024 regressed after the overlay, so another broad participation floor is not justified.',
        'Both remaining failures are relative-return gaps; drawdown is below the 20pct gate for the selected candidates.',
        'Train target exposure is much higher than selected test exposure, especially share >=50pct, indicating state-transfer or representation failure rather than a completed target handoff.',
    ],
}
summary = dict(selection)
summary['fold_gate_gap_summary'] = fold_gate_gap.to_dict('records')
summary['missed_up_days_summary'] = missed_summary.to_dict('records')
summary['target_to_selected_exposure_gap'] = exposure_gap.to_dict('records')
summary['test_state_trigger_overlap'] = test_state_overlap.to_dict('records')
summary['bc_representation_summary'] = bc_representation.to_dict('records')
summary['conclusion'] = 'Do not promote or retain. Mine failed-fold test-state features and rebuild the representation/reward objective before another training branch.'

write_json(SIDE_CARS['selection'], selection)
write_json(SIDE_CARS['summary'], summary)
print(json.dumps(selection, indent=2, ensure_ascii=False)[:5000])


{
  "experiment": "release16_remaining_failedfold_diagnostic_v1",
  "created_at": "2026-06-14T15:29:22.833088+00:00",
  "decision": "release16_overlay_failure_is_state_transfer_and_representation_gap_no_promotion",
  "canonical_baseline": "retained_soxl_harness_model_frontier",
  "direct_comparison_baseline": "release16_failedfold_overlay_checkpoint_screen_v1",
  "source_checkpoint_screen": "release16_failedfold_overlay_checkpoint_screen_v1",
  "source_target_experiment": "release16_failedfold_participation_target_overlay_v1",
  "reference_baseline": "soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02",
  "role": "diagnose",
  "changed_layer": [
    "model_selection",
    "failure_attribution",
    "target_to_replay_diagnostic"
  ],
  "target_window": [
    "wf_2020",
    "wf_2023_2024"
  ],
  "target_relative_return_pp": 30.0,
  "target_max_dd_pct": 20.0,
  "failed_folds": [
    "wf_2020",
    "wf_2023_2024"
  ],
  "guard_folds": [
    "wf_2022"
  ],
  "metrics": [
    {

In [10]:
selection = load_json(SIDE_CARS['selection'])
fold_gate_gap = pd.read_csv(SIDE_CARS['fold_gate_gap_summary'])
missed_summary = pd.read_csv(SIDE_CARS['missed_up_days_summary'])
exposure_gap = pd.read_csv(SIDE_CARS['target_to_selected_exposure_gap'])
test_state_overlap = pd.read_csv(SIDE_CARS['test_state_trigger_overlap'])
bc_representation = pd.read_csv(SIDE_CARS['bc_representation_summary'])

lines = [
    '# Release16 Remaining Failed-Fold Diagnostic',
    '',
    f"Decision: `{selection['decision']}`.",
    '',
    'Gate gaps:',
]
for row in fold_gate_gap[fold_gate_gap['fold'].isin(FAILED_FOLDS)].itertuples(index=False):
    lines.append(
        f"- {row.fold}: rel {row.selected_relative_return_pp:.2f}pp, return gap to +30pp relative {row.return_gap_to_plus30_relative_pp:.2f}pp, "
        f"MaxDD {row.selected_max_dd_pct:.2f}%, overlay relative delta {row.overlay_relative_delta_pp:+.2f}pp."
    )
lines.extend(['', 'Exposure transfer:'])
for row in exposure_gap[exposure_gap['fold'].isin(FAILED_FOLDS)].itertuples(index=False):
    lines.append(
        f"- {row.fold}: train target mean {row.target_mean_pos:.3f} vs selected test mean {row.selected_mean_pos:.3f}; "
        f"target share >=50% {row.target_share_ge50:.3f} vs selected share >=50% {row.selected_share_ge50:.3f}."
    )
lines.extend(['', 'Missed low-position up days:'])
for row in missed_summary.itertuples(index=False):
    lines.append(
        f"- {row.fold}: {int(row.days)} days, mean prev position {row.mean_selected_prev_position:.3f}, "
        f"SOXL up-day sum {row.sum_soxl_up_day_return_pp:.2f}pp, 50% floor counterfactual {row.sum_incremental_if_50pct_pp:.2f}pp."
    )
lines.extend(['', 'State/representation notes:'])
for row in test_state_overlap.itertuples(index=False):
    lines.append(
        f"- {row.fold}: proxy trigger days {int(row.trigger_days)} / {int(row.test_rows)}, low-position up-day trigger overlap {int(row.low_position_up_days_trigger_overlap)} / {int(row.low_position_up_days)}."
    )
for row in bc_representation[bc_representation['fold'].isin(FAILED_FOLDS)].itertuples(index=False):
    lines.append(
        f"- {row.fold}: BC all accuracy {row.bc_all_accuracy:.3f}, label dominant share {row.label_dominant_share:.3f}, predicted dominant share {row.predicted_dominant_share:.3f}, selected-minus-prior avg pos {row.selected_minus_prior_mean_pos:+.3f}."
    )
lines.extend([
    '',
    'Risk note: this diagnostic does not train or retain an actor. It uses test-window daily paths only for failure attribution, not for target generation.',
    '',
    f"Next action: `{selection['next_mechanism']}`.",
    '',
    'Artifacts: `fold_gate_gap_summary.csv`, `monthly_overlay_vs_prior_vs_reference.csv`, `missed_up_days_after_overlay.csv`, `missed_up_days_summary.csv`, `target_to_selected_exposure_gap.csv`, `test_state_trigger_overlap.csv`, `bc_representation_summary.csv`, `selection.json`, `summary.json`, `browser_dashboard_kimi_runbacktest.png`.',
])
conclusion = '\n'.join(lines).rstrip() + '\n'
SIDE_CARS['conclusion'].write_text(conclusion, encoding='utf-8')
display(Markdown(conclusion))


# Release16 Remaining Failed-Fold Diagnostic

Decision: `release16_overlay_failure_is_state_transfer_and_representation_gap_no_promotion`.

Gate gaps:
- wf_2020: rel 11.42pp, return gap to +30pp relative 18.58pp, MaxDD 16.65%, overlay relative delta +26.45pp.
- wf_2023_2024: rel 3.41pp, return gap to +30pp relative 26.59pp, MaxDD 5.55%, overlay relative delta -1.30pp.

Exposure transfer:
- wf_2020: train target mean 0.364 vs selected test mean 0.230; target share >=50% 0.302 vs selected share >=50% 0.071.
- wf_2023_2024: train target mean 0.237 vs selected test mean 0.083; target share >=50% 0.300 vs selected share >=50% 0.000.

Missed low-position up days:
- wf_2020: 47 days, mean prev position 0.102, SOXL up-day sum 376.80pp, 50% floor counterfactual 150.03pp.
- wf_2023_2024: 66 days, mean prev position 0.070, SOXL up-day sum 490.29pp, 50% floor counterfactual 211.57pp.

State/representation notes:
- wf_2020: proxy trigger days 13 / 252, low-position up-day trigger overlap 5 / 47.
- wf_2023_2024: proxy trigger days 141 / 251, low-position up-day trigger overlap 41 / 66.
- wf_2020: BC all accuracy 0.738, label dominant share 0.352, predicted dominant share 0.457, selected-minus-prior avg pos +0.006.
- wf_2023_2024: BC all accuracy 0.709, label dominant share 0.462, predicted dominant share 0.573, selected-minus-prior avg pos -0.002.

Risk note: this diagnostic does not train or retain an actor. It uses test-window daily paths only for failure attribution, not for target generation.

Next action: `mine_failedfold_test_state_features_then_rebuild_representation_or_reward_objective_before_training`.

Artifacts: `fold_gate_gap_summary.csv`, `monthly_overlay_vs_prior_vs_reference.csv`, `missed_up_days_after_overlay.csv`, `missed_up_days_summary.csv`, `target_to_selected_exposure_gap.csv`, `test_state_trigger_overlap.csv`, `bc_representation_summary.csv`, `selection.json`, `summary.json`, `browser_dashboard_kimi_runbacktest.png`.
